In [7]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Libraries imported successfully.")

Libraries imported successfully.


In [8]:
df = pd.read_parquet("../dataset/BGL_Preprocessed.parquet")

print(df.shape)
df.head()

(4747963, 20)


,Label,Timestamp,Date,Node,Time,NodeRepeat,Type,Component,Severity,Message,Message_Length,Clean_Message,Word_Count,Has_Number,Has_Hex,Has_Path,Keyword_Count,Unique_Word_Count,Uppercase_Ratio,Incident_Type
0,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.363779,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,40,instruction cache parity error corrected,5,False,False,False,1,5,0.0,Memory Error
1,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.527847,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,40,instruction cache parity error corrected,5,False,False,False,1,5,0.0,Memory Error
2,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.675872,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,40,instruction cache parity error corrected,5,False,False,False,1,5,0.0,Memory Error
3,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.823719,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,40,instruction cache parity error corrected,5,False,False,False,1,5,0.0,Memory Error
4,-,1117838570,2005.06.03,R02-M1-N0-C:J12-U11,2005-06-03-15.42.50.982731,R02-M1-N0-C:J12-U11,RAS,KERNEL,INFO,instruction cache parity error corrected,40,instruction cache parity error corrected,5,False,False,False,1,5,0.0,Memory Error


In [9]:
df.columns

Index(['Label', 'Timestamp', 'Date', 'Node', 'Time', 'NodeRepeat', 'Type',
       'Component', 'Severity', 'Message', 'Message_Length', 'Clean_Message',
       'Word_Count', 'Has_Number', 'Has_Hex', 'Has_Path', 'Keyword_Count',
       'Unique_Word_Count', 'Uppercase_Ratio', 'Incident_Type'],
      dtype='object')

In [10]:
RULES = {
    "Memory Error": [
        "ddr", "edram", "memory", "cache parity", "tlb", "parity", "cache"
    ],
    "Application Failure": [
        "assert", "exception", "signal", "core dump", "segmentation"
    ],
    "Hardware Failure": [
        "node card", "vpd", "assembly", "jtag", "pgood", "mpgood", "asic"
    ],
    "Deployment Error": [
        "error loading", "program image", "exec format", "invalid or missing program"
    ],
    "System Interrupt": [
        "interrupt", "machine check"
    ],
    "Network/Communication": [
        "network", "packet", "transport", "message prefix", "ciostream", "connection"
    ],
    "Storage/File System": [
        "mount", "filesystem", "lustre", "directory", "permission", "file", "no such file"
    ],
    "Power/Thermal Issue": [
        "temperature", "thermal", "volt", "voltage", "rail", "clock"
    ]
}

def assign_incident_type(message):
    msg = str(message).lower()

    for incident_type, keywords in RULES.items():
        if any(keyword in msg for keyword in keywords):
            return incident_type

    return "Other"

In [11]:
df["Incident_Type"] = df["Message"].apply(assign_incident_type)

df[["Message", "Clean_Message", "Incident_Type"]].head(10)

,Message,Clean_Message,Incident_Type
0,instruction cache parity error corrected,instruction cache parity error corrected,Memory Error
1,instruction cache parity error corrected,instruction cache parity error corrected,Memory Error
2,instruction cache parity error corrected,instruction cache parity error corrected,Memory Error
3,instruction cache parity error corrected,instruction cache parity error corrected,Memory Error
4,instruction cache parity error corrected,instruction cache parity error corrected,Memory Error
5,instruction cache parity error corrected,instruction cache parity error corrected,Memory Error
6,instruction cache parity error corrected,instruction cache parity error corrected,Memory Error
7,instruction cache parity error corrected,instruction cache parity error corrected,Memory Error
8,instruction cache parity error corrected,instruction cache parity error corrected,Memory Error
9,instruction cache parity error corrected,instruction cache parity error corrected,Memory Error


In [12]:
incident_counts = df["Incident_Type"].value_counts()
incident_counts

Incident_Type
Other                    2613363
Application Failure       725524
Memory Error              637479
System Interrupt          368414
Storage/File System       157848
Deployment Error          115032
Network/Communication      71004
Hardware Failure           58322
Power/Thermal Issue          977
Name: count, dtype: int64

In [13]:
df["Incident_Type"].value_counts(normalize=True).mul(100).round(2)

Incident_Type
Other                    55.04
Application Failure      15.28
Memory Error             13.43
System Interrupt          7.76
Storage/File System       3.32
Deployment Error          2.42
Network/Communication     1.50
Hardware Failure          1.23
Power/Thermal Issue       0.02
Name: proportion, dtype: float64

In [14]:
df[df["Incident_Type"] == "Other"]["Clean_Message"].sample(20, random_state=42)

3779376              iar ea dear c c
3982292               iar a dear f c
776184               generating core
4069505              iar e dear cb c
2310807              generating core
4061181             iar e dear f f c
1118869              generating core
2319797              generating core
2155698              generating core
705597           eeeeeee ffffffff cf
2828476              generating core
2782569              generating core
1495784              generating core
4030885                 iar dear a c
1215759              generating core
3387966               ce sym at mask
4059556                 iar dear c c
198894               generating core
74571      general purpose registers
3006585               ce sym at mask
Name: Clean_Message, dtype: object

In [15]:
df_model = df[
    (df["Clean_Message"].notna()) &
    (df["Clean_Message"].str.strip() != "") &
    (df["Incident_Type"].notna())
].copy()

print(df_model.shape)
df_model[["Clean_Message", "Incident_Type"]].head()

(4710847, 20)


,Clean_Message,Incident_Type
0,instruction cache parity error corrected,Memory Error
1,instruction cache parity error corrected,Memory Error
2,instruction cache parity error corrected,Memory Error
3,instruction cache parity error corrected,Memory Error
4,instruction cache parity error corrected,Memory Error


In [16]:
df_model.to_parquet("../dataset/BGL_Preprocessed.parquet", index=False)

print("Updated preprocessed dataset saved successfully.")

Updated preprocessed dataset saved successfully.
